# Week 11 Lab. Filter, Break the Sensor, Then Smooth
**Time Series Analysis & Random Processes** · Graduate School of Data Science, Chonnam National University · notebook v2 (2026-11-10)

---

### 이 실습에서 하는 일 / What this lab does

오늘 덱의 센서를 그대로 만듭니다. 1분에 한 번 값을 읽는 센서 120분, 참값은 조금씩 움직이고 관측마다 잡음이 섞입니다. 참값을 알고 있으므로 추측이 얼마나 맞는지 잴 수 있습니다. 필터의 두 걸음(예측과 갱신)을 세 줄로 직접 쓰고, 센서를 15분 동안 끊어 보고, 마지막에 스무더와 비교합니다. 단계마다 **먼저 예측하고** 실행합니다.
*We rebuild the sensor from today's deck: 120 minutes, one reading a minute. The true value moves a little each minute and every reading carries noise. Because we know the true value, we can measure how good each guess is. You write the filter's two steps (predict and update) in three lines, break the sensor for 15 minutes, and compare with the smoother at the end. Every step is **predict first**, then run.*

1. **모든 관측에 잡음이 있다면, 지금 참값의 가장 좋은 추측은 무엇인가? / If every reading is noisy, what is our best guess of the true value right now?**
2. **새 관측이 오면 추측을 얼마나 옮겨야 하나? / When a new reading arrives, how far should our guess move?**
3. **관측이 오지 않으면 어떻게 해야 하나? / What should we do when some readings never arrive?**

덱 「Three Questions for Today」 의 세 질문입니다. 1단계가 1번, 2 ~ 5단계가 2번, 4 · 6단계가 3번에 답합니다. 7 · 8단계는 덱 「In the Lab, Filter, Break the Sensor, Then Smooth」 표의 마지막 두 줄입니다.
*These are the three questions on "Three Questions for Today". Step 1 answers question 1, steps 2 to 5 answer question 2, steps 4 and 6 answer question 3. Steps 7 and 8 are the last two rows of the table on "In the Lab, Filter, Break the Sensor, Then Smooth".*

| 단계 / Step | 먼저 예측 / Before running, predict | 실행하고 확인 / Then run and check | 짝 슬라이드 / Matching slides |
|---|---|---|---|
| 1 | 최근 관측과 지금까지의 평균, 어느 쪽이 참값에 가까울까?<br>The latest reading or the average so far: which is closer to the truth? | 센서를 만들고 단순한 추측 둘의 오차를 잰다<br>Make the sensor and score two simple guesses | 「What Is the True Value Behind These Readings?」 · 「Why Not Use the Latest Reading or the Average?」 |
| 2 | 센서가 시끄러워지면 이득은 커질까 작아질까?<br>Does the gain go up or down when the sensor gets noisier? | 이득 한 줄(TODO 1)<br>The gain line (TODO 1) | 「A Noisy Sensor Gets a Small Weight」 · 「Why Is 3/7 the Best Weight?」 |
| 3 | 예측 10, 관측 14, 이득 3/7 이면 새 추측은?<br>Prediction 10, reading 14, gain 3/7: what is the new guess? | 추측을 옮기는 한 줄(TODO 2)<br>The line that moves the guess (TODO 2) | 「The Reading Is 14. Where Should Our Guess Go?」 |
| 4 | 관측이 오지 않으면 불확실성은 어떻게 되나?<br>What happens to the uncertainty when no reading arrives? | 한 걸음 함수와 결측 한 줄(TODO 3)<br>The one-step function and the missing-reading line (TODO 3) | 「The Whole Filter Is Two Steps, Repeated」 · 「What If a Reading Never Arrives?」 |
| 5 | 첫 이득은 0 근처일까 1 근처일까? 이득은 어디에 자리 잡을까?<br>Is the first gain near 0 or near 1? Where will the gain settle? | 처음 세 이득을 찍고 이득을 그린다<br>Print the first three gains and plot the gain | 「Watch the Filter Track the Hidden Value」 · 「How Quickly Does the Gain Settle?」 |
| 6 | 걸음 45 ~ 59 에서 띠는 어떻게 되나?<br>What happens to the band in steps 45 to 59? | 관측 45 ~ 59 를 지우고 필터와 95 % 띠를 그린다<br>Remove readings 45 to 59, plot the filter with its 95% band | 「The Filter Across a Gap in the Readings」 |
| 7 | 관측 · 필터 · 스무더 가운데 무엇이 참값에 가장 가까울까?<br>Which is closest to the truth: readings, filter or smoother? | 스무더와 로그우도(읽기만), 오차 비교<br>The smoother and the log-likelihood (read only), then compare errors | 「What Else Does the Same Recursion Give?」 |
| 8 | 센서가 더 시끄러우면 자리 잡는 이득은?<br>What if the sensor were noisier: where does the gain settle? | 설정 하나를 바꿔 다시 돌린다<br>Change one choice and rerun | 「In the Lab, Filter, Break the Sensor, Then Smooth」 · 「The Settled Gain in One Formula」 |

### 쓰는 법 / How to use it

**`TODO` 가 세 개 있습니다.** 각 `TODO` 는 `… = None` 한 줄입니다. `None` 자리에 코드를 넣으면 되고, 지울 줄은 없습니다. 채우기 전에 실행하면 `NotImplementedError` 와 안내문이 뜨며 멈춥니다. **고장이 아니라 정상입니다.** `TODO` 바로 아래의 **자가검증** 셀이 내 답을 검사하고, 그 아래에 접힌 **Solution (정답)** 셀이 있습니다. 먼저 해 본 뒤에만 여세요. 정답 셀을 돌렸다면 자가검증 셀을 다시 실행하세요.
*There are **three `TODO`s**. Each is one line that reads `… = None`. Replace the `None` with your code; nothing needs deleting. If you run the cell before that, it stops with a `NotImplementedError` and a message. **That is expected, not a broken notebook.** The **self-check** cell right below each `TODO` tests your answer, and below it is a collapsed **Solution (정답)** cell. Open it only after you have tried. If you ran the Solution cell, run the self-check again.*

> ### 먼저 **파일 > Drive에 사본 저장** / First, **File > Save a copy in Drive**
> 올려 둔 링크는 **읽기 전용**으로 열립니다. 사본을 먼저 저장하지 않으면 입력한 내용이 탭을 닫을 때 사라집니다.
> *The posted link opens **read-only**. Save your own copy first, or everything you type disappears when you close the tab.*
>
> **제출하지 않고 채점하지 않습니다.** 11주차 실습 시간(약 35분)용입니다. 시간이 모자라면 8단계와 「더 해 보기」 는 집에서 하세요.
> *Not submitted, not graded. This is the Week 11 lab block (about 35 minutes). If time runs out, do step 8 and the optional part at home.*

**자료.** 덱의 그림과 숫자는 이 노트북과 같은 모의에서 나왔습니다: 시드 11, 120걸음, 참값이 한 걸음에 움직이는 표준편차 0.3, 센서 잡음의 표준편차 1, 그리고 관측 45 ~ 59 를 지운 것. 그래서 **덱의 숫자와 이 노트북의 숫자가 같아야 정상입니다.** 덱에서 「IN THE LAB」 표시가 붙은 장이 이 노트북의 어느 단계 · 어느 TODO 와 짝인지 알려 줍니다. GPU 가 없어도 되고 설치할 것도 없습니다. 전체 실행은 몇 초입니다.
*Data. The deck's figures and numbers come from the same simulation as this notebook: seed 11, 120 steps, the truth moving with standard deviation 0.3 a step, sensor noise with standard deviation 1, and readings 45 to 59 removed. So **the deck's numbers and this notebook's numbers should be the same.** On the deck, the slides marked "IN THE LAB" tell you which step or which TODO of this notebook they pair with. No GPU and nothing to install; the whole notebook runs in a few seconds.*

---
### 준비 / Setup

설치할 것은 없습니다. NumPy 와 Matplotlib 만 쓰고, 둘 다 Colab 에 이미 있습니다.
*Nothing to install: only NumPy and Matplotlib, and both ship with Colab.*

In [ ]:
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
DARK, GREEN, GOLD, GRAY, RED = "#11241C", "#1E4A32", "#D9AE3A", "#9A9A9A", "#B23A2E"   # 덱과 같은 색 / the deck's colors

# 빈칸(TODO)을 아직 안 채웠을 때 보여 줄 안내입니다. 고장이 아니라는 뜻입니다.
# Message shown when a TODO is still blank. It means the notebook is fine, not broken.
BLANK_MSG = (
    "\n\n"
    "  정상입니다. 고장이 아니라 일부러 비워 둔 칸입니다.\n"
    "     위의 None 자리에 코드를 넣고 이 셀을 다시 실행하세요. 지울 줄은 없습니다.\n"
    "     막힐 때 자가검증 셀 아래의 'Solution / 정답' 셀을 실행하고, 자가검증 셀을 다시 실행하면 이어서 진행됩니다.\n\n"
    "  This is expected, not a broken notebook.\n"
    "     Replace None above with your code and re-run. Nothing needs deleting.\n"
    "     If stuck, run the Solution cell below the self-check, then run the self-check again and continue.\n"
)
print("numpy", np.__version__, "· matplotlib", matplotlib.__version__)
# 기대값 / Expected   판 번호는 컴퓨터마다 달라도 됩니다. 아래 기대값은 numpy 2.5 에서 찍었습니다.
#   Version numbers may differ by computer. The expected values below come from numpy 2.5.

**덱의 기호와 이 노트북의 이름 / The deck's symbols and this notebook's names.** 덱 「Same Model, Three Notations」 의 「Lab code」 칸과 같습니다.
*The same as the "Lab code" column of "Same Model, Three Notations".*

| 뜻 / Idea | 덱 / This deck | 이 노트북 / This notebook |
|---|---|---|
| 숨은 값(상태) / Hidden value (state) | $\alpha_t$ | `alpha` |
| 관측 / Reading (observation) | $y_t$ | `y`, 끊긴 센서는 `y_gap`<br>`y`, and `y_gap` for the broken sensor |
| 참값이 움직이는 정도 / How much the truth moves | $Q$ | `s_eta2` = $0.3^2$ = 0.09 |
| 센서가 시끄러운 정도 / How noisy the sensor is | $H$ | `s_eps2` = $1^2$ = 1 |
| 관측을 보기 전 / Before the reading | $a^-,\ P^-$ | `a_pred`, `P_pred` |
| 관측을 본 뒤 / After the reading | $a,\ P$ | `a_filt`, `P_filt` |
| 이득 / Gain | $K$ | `K` |

**준비 셀이 만든 도구 / Tools made by the next cell.** 아래 접힌 셀을 그대로 실행하세요. 외울 것은 없습니다. 무엇을 계산하는지만 보세요. 필터 자체는 여기에 없습니다. 2 ~ 4단계에서 여러분이 세 줄을 채워 만듭니다.
*Run the collapsed cell below as it is. Nothing to memorize; just know what each tool computes. The filter itself is not in there: you build it in steps 2 to 4 by filling in three lines.*

| 이름 / Name | 하는 일 / What it does |
|---|---|
| `simulate_sensor(seed, n, sigma_eta, sigma_eps)` | 참값 `alpha` 와 관측 `y`, 각각 `n` 걸음. 참값은 10 에서 출발해 걸음마다 조금씩 움직인다<br>The true value `alpha` and the readings `y`, `n` steps each. The truth starts at 10 and moves a little each step |
| `settled_gain(s_eta2, s_eps2)` | 이득이 자리 잡는 값을 식으로(덱 「The Settled Gain in One Formula」)<br>Where the gain settles, by formula ("The Settled Gain in One Formula") |
| `kalman_smoother(a_pred, P_pred, a_filt, P_filt)` | 필터 결과를 뒤에서 앞으로 한 번 더 지나가며 나중 관측까지 쓴 추측을 만든다. 7단계에서 읽기만<br>One backward pass over the filter output, giving guesses that also use later readings. Read only, in step 7 |
| `log_likelihood(y, a_pred, P_pred, s_eps2)` | 놀람들의 로그 밀도를 더한 값(첫 관측 제외). 7단계에서 읽기만<br>The sum of the surprises' log densities (first reading left out). Read only, in step 7 |
| `rmse(guess, truth)` | 추측과 참값 사이 오차의 크기(오차 제곱 평균의 제곱근)<br>The size of the error between a guess and the truth (root mean squared error) |
| `run_lab(sigma_eps, gap, seed)` | 설정 하나만 바꿔 1 ~ 7단계를 다시(8단계)<br>Steps 1 to 7 again with one setting changed (step 8) |

In [ ]:
#@title 도구 / Tools (그대로 실행 / run as it is) { display-mode: "form" }
def simulate_sensor(seed=11, n=120, sigma_eta=0.3, sigma_eps=1.0):
    """참값 alpha 와 관측 y, 각각 길이 n / the true value alpha and the readings y, each of length n."""
    rng = np.random.default_rng(seed)                 # 시드를 고정한 난수 생성기 / a random number generator with a fixed seed
    eta = rng.normal(0.0, sigma_eta, n)               # 걸음마다 참값이 움직이는 양 / how far the truth moves at each step
    alpha = 10 + np.cumsum(eta)                       # 참값: 10 에서 출발해 움직임을 쌓는다(5주차 랜덤워크) / the truth: start at 10, add up the moves (Week 5's random walk)
    eps = rng.normal(0.0, sigma_eps, n)               # 센서 잡음 / the sensor noise
    return alpha, alpha + eps                         # 관측 = 참값 + 잡음 / reading = truth + noise


def settled_gain(s_eta2, s_eps2):
    """이득이 자리 잡는 값 / where the gain settles (deck: "The Settled Gain in One Formula")."""
    # 자리 잡으면 P_pred 가 걸음마다 같다: (P_pred)^2 − Q × P_pred − Q × H = 0. 그 양수 해.
    # Once settled, P_pred is the same at every step: (P_pred)^2 − Q × P_pred − Q × H = 0. Its positive root.
    P_pred = (s_eta2 + np.sqrt(s_eta2 ** 2 + 4 * s_eta2 * s_eps2)) / 2
    return P_pred / (P_pred + s_eps2)                 # 이득 한 줄 / the gain line


def kalman_smoother(a_pred, P_pred, a_filt, P_filt):
    """뒤에서 앞으로 한 번 더 / one more pass, backward: guesses that also use later readings."""
    n = len(a_filt)
    a_smooth, P_smooth = a_filt.copy(), P_filt.copy()        # 마지막 걸음은 필터와 같다 / the last step equals the filter
    for t in range(n - 2, -1, -1):                           # 끝에서 둘째 걸음부터 처음까지 / from the second-to-last step back to the first
        J = P_filt[t] / P_pred[t + 1]                        # 다음 걸음에서 얼마나 가져올지 / how much to borrow from the next step
        a_smooth[t] = a_filt[t] + J * (a_smooth[t + 1] - a_pred[t + 1])        # 다음 걸음이 예측과 달랐던 만큼 고친다 / correct by how far the next step differed from its prediction
        P_smooth[t] = P_filt[t] + J ** 2 * (P_smooth[t + 1] - P_pred[t + 1])   # 불확실성도 그만큼 줄어든다 / the uncertainty shrinks accordingly
    return a_smooth, P_smooth


def log_likelihood(y, a_pred, P_pred, s_eps2):
    """놀람들의 로그 밀도의 합, 첫 관측 제외 / the sum of the surprises' log densities, first reading left out."""
    seen = ~np.isnan(y)                                      # 관측이 있는 걸음 / the steps with a reading
    seen[np.argmax(seen)] = False                            # 첫 관측은 뺀다. 시작 P0 가 아주 커서 그 항은 P0 만 반영한다 / drop the first one: with a huge starting P0 its term only reflects P0
    v = y[seen] - a_pred[seen]                               # 놀람 = 관측 − 예측 / surprise = reading − prediction
    F = P_pred[seen] + s_eps2                                # 놀람의 분산 = 예측의 분산 + 센서의 분산 / the surprise's variance = prediction variance + sensor variance
    return float(-0.5 * np.sum(np.log(2 * np.pi * F) + v ** 2 / F))   # 정규분포 로그 밀도의 합 / the sum of normal log densities


def rmse(guess, truth):
    """오차 제곱 평균의 제곱근 / root mean squared error."""
    return float(np.sqrt(np.mean((np.asarray(guess) - np.asarray(truth)) ** 2)))


def run_lab(sigma_eps=1.0, gap=(45, 60), seed=11):
    """설정 하나를 바꿔 1 ~ 7단계를 다시 / steps 1 to 7 again with one setting changed."""
    alpha, y = simulate_sensor(seed=seed, sigma_eps=sigma_eps)            # 새 센서 / a new sensor
    y_gap = y.copy()
    y_gap[gap[0]:gap[1]] = np.nan                                         # 관측을 지운다 / remove the readings
    a_pred, P_pred, K, a_filt, P_filt = kalman_filter(y_gap, 0.3 ** 2, sigma_eps ** 2)   # 5단계의 필터 / step 5's filter
    a_smooth, _ = kalman_smoother(a_pred, P_pred, a_filt, P_filt)
    seen = ~np.isnan(y_gap)
    return (K[gap[0] - 1], K[gap[1]], 1.96 * np.sqrt(P_filt[gap[1] - 1]),                # 자리 잡은 이득, 결측 뒤 첫 이득, 결측 마지막 시점의 띠 반폭 / settled gain, first gain after the gap, half-width at the gap end
            rmse(y_gap[seen], alpha[seen]), rmse(a_filt, alpha), rmse(a_smooth, alpha))  # 오차: 관측, 필터, 스무더 / errors: readings, filter, smoother

---
### 1단계. 센서를 만들고 단순한 추측 둘을 재기 / Step 1. Make the sensor and score two simple guesses

이 실습은 가장 단순한 상태공간모형인 **지역수준모형(local level model)**을 씁니다. 숨은 수준은 랜덤워크로 움직이고, 관측은 그 수준에 센서 잡음이 더해진 값입니다.
*We use the **local level model**, the simplest state-space model: the hidden level follows a random walk, and each reading adds measurement noise.*

덱 「What Is the True Value Behind These Readings?」: 센서는 1분에 한 번 값을 보고하고, 관측은 참값에 잡음이 더해진 것입니다. 참값은 한 걸음에 표준편차 0.3 만큼 움직이고(분산 `s_eta2` = 0.09, 덱의 $Q$), 센서 잡음의 표준편차는 1 입니다(분산 `s_eps2` = 1, 덱의 $H$).
*"What Is the True Value Behind These Readings?": the sensor reports once a minute, and each reading is the true value plus noise. The truth moves with standard deviation 0.3 a step (variance `s_eta2` = 0.09, $Q$ on the deck), and the sensor noise has standard deviation 1 (variance `s_eps2` = 1, $H$ on the deck).*

**코드를 돌리기 전에 예측하세요.** 처음 45분 동안 「가장 최근 관측」 과 「지금까지 관측의 평균」 가운데 어느 쪽이 참값에 더 가까울까요?
***Predict before you run.** Over the first 45 minutes, which is closer to the true value: the latest reading, or the average of all readings so far?*

In [ ]:
sigma_eta, sigma_eps = 0.3, 1.0                            # 참값이 움직이는 표준편차, 센서 잡음의 표준편차 / SD of the truth's moves, SD of the sensor noise
s_eta2, s_eps2 = sigma_eta ** 2, sigma_eps ** 2            # 분산 = 표준편차의 제곱. 덱의 Q = 0.09, H = 1 / variance = SD squared: the deck's Q = 0.09 and H = 1
alpha, y = simulate_sensor(seed=11, n=120, sigma_eta=sigma_eta, sigma_eps=sigma_eps)   # 참값과 관측, 120걸음 / truth and readings, 120 steps
first = slice(0, 45)                                       # 처음 45분 = 걸음 0 ~ 44 / the first 45 minutes = steps 0 to 44
average_so_far = np.cumsum(y) / np.arange(1, len(y) + 1)   # 걸음마다 그때까지 관측의 평균 / at each step, the mean of the readings so far
print("참값 / true value:", np.round(alpha[:3], 4), "  관측 / readings:", np.round(y[:3], 4))
print(f"처음 45분의 오차(RMSE) / error over the first 45 minutes (RMSE):  최근 관측 / latest reading {rmse(y[first], alpha[first]):.3f},"
      f"  지금까지의 평균 / average so far {rmse(average_so_far[first], alpha[first]):.3f}")
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(alpha[first], color=DARK, lw=2, label="true value")
ax.plot(y[first], "o", color=GRAY, ms=4, label="readings")
ax.plot(average_so_far[first], "--", color=RED, label="average so far")
ax.set(xlabel="step t (minutes)", ylabel="value", title="We see only the readings; the true value stays hidden")   # 그림 글은 영어만 / plot text in English only
ax.legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show()
# 기대값 / Expected   참값 / true value: [10.0103 10.4182 10.7856]   관측 / readings: [10.6064  9.5878 10.515 ]
# 기대값 / Expected   최근 관측 / latest reading 0.928,  지금까지의 평균 / average so far 0.827
#   덱 「Why Not Use the Latest Reading or the Average?」 의 표와 같은 숫자입니다. 다르게 나오면 맨 아래 환경 기록 셀의 출력을 게시판에 올려 주세요.
#   The same numbers as the table on "Why Not Use the Latest Reading or the Average?". If yours differ, post the output of the environment cell at the bottom on the board.

**What to notice.** 최근 관측은 잡음을 그대로 지고 있고(오차 0.928), 평균은 잡음을 지우지만 참값이 움직이면 뒤처집니다(0.827). 그 사이의 무언가가 필요합니다: 새 관측 쪽으로 **일부만** 움직이는 추측입니다. 얼마나 움직일지가 2단계의 이득입니다.
*The latest reading carries all of its noise (error 0.928). The average smooths the noise away but falls behind whenever the true value moves (0.827). We need something in between: a guess that moves only **part of the way** toward each new reading. How far is the gain of step 2.*

---
### 2단계. 이득 한 줄 / Step 2. The gain line

덱 「A Noisy Sensor Gets a Small Weight」: 새 관측을 얼마나 믿을지는 두 불확실성을 견주어 정합니다. 예측의 오차 분산이 `P_pred`(덱의 $P^-$), 센서 잡음의 분산이 `s_eps2`(덱의 $H$)일 때 이득은 전체 불확실성 가운데 예측이 차지하는 몫입니다.
*"A Noisy Sensor Gets a Small Weight": how much to trust a new reading comes from comparing two uncertainties. With the prediction's error variance `P_pred` ($P^-$ on the deck) and the sensor's noise variance `s_eps2` ($H$ on the deck), the gain is the prediction's share of the total uncertainty.*

$$K = \frac{P^-}{P^- + H}$$

**TODO 1** 은 이 한 줄입니다. 손계산 예: $P^- = 3$, $H = 4$ 이면 $K = 3/7 \approx 0.43$.
***TODO 1** is this line. By hand: with $P^- = 3$ and $H = 4$, $K = 3/7 \approx 0.43$.*

**코드를 돌리기 전에 예측하세요.** 센서가 더 시끄러워지면($H$ 가 커지면) 이득은 커질까요, 작아질까요?
***Predict before you run.** If the sensor gets noisier (a larger $H$), does the gain go up or down?*

In [ ]:
# ---------------------------------------------------------------- TODO 1 ----
# 다음 이름을 씁니다 / Names used below
#   P_pred  : 관측을 보기 전, 예측의 오차 분산(덱에서 − 가 붙은 P) / the prediction's error variance, before the reading (the P with a minus sign on the deck)
#   s_eps2  : 센서 잡음의 분산(덱의 H) / the sensor's noise variance (H on the deck)
#   K       : 이득, 0 과 1 사이의 숫자   <- 여기가 빈칸
#             the gain, a number between 0 and 1   <- this is the blank


def kalman_gain(P_pred, s_eps2):
    """덱 「A Noisy Sensor Gets a Small Weight」 의 한 줄 / the one line on that slide."""
    # TODO 1. 전체 불확실성(예측의 분산 + 센서의 분산) 가운데 예측이 차지하는 몫을 쓰세요. 한 줄입니다.
    # TODO 1. Write the prediction's share of the total uncertainty (prediction variance + sensor variance). One line.
    K = None

    if K is None:
        raise NotImplementedError(BLANK_MSG)

    return K

In [ ]:
# 자가검증 / Self-check   덱 「A Noisy Sensor Gets a Small Weight」 의 표 / the table on that slide
got = kalman_gain(3, 4)
assert np.isclose(got, 3 / 7), (
    f"P_pred = 3, s_eps2 = 4 에서 정답은 3/7 = 0.4286, 결과는 {got:.4f} 입니다. "
    "0.5714 면 센서의 몫(s_eps2 를 분자에)을 썼고, 0.75 면 P_pred / s_eps2 를 썼습니다.\n"
    f"With P_pred = 3 and s_eps2 = 4 the answer is 3/7 = 0.4286, but got {got:.4f}. "
    "0.5714 means the sensor's share (s_eps2 on top); 0.75 means P_pred / s_eps2.")
gains = [round(float(kalman_gain(3, H)), 2) for H in (0.5, 4, 40)]
assert gains == [0.86, 0.43, 0.07], f"덱의 표는 0.86, 0.43, 0.07 인데 결과는 {gains} 입니다 / the deck's table has 0.86, 0.43, 0.07 but got {gains}."
print("통과. 센서 잡음의 분산이 0.5, 4, 40 일 때 이득은", gains, ": 덱의 표입니다. 센서가 시끄러울수록 이득이 작습니다.")
print("Passed. With sensor noise variance 0.5, 4, 40 the gains are", gains, ": the deck's table. The noisier the sensor, the smaller the gain.")
# 기대값 / Expected   [0.86, 0.43, 0.07]

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.


def kalman_gain(P_pred, s_eps2):
    """덱 「A Noisy Sensor Gets a Small Weight」 의 한 줄 / the one line on that slide."""
    K = P_pred / (P_pred + s_eps2)                   # 빈칸이었던 줄. 전체 가운데 예측의 몫 / the blank: the prediction's share of the total

    if K is None:
        raise NotImplementedError(BLANK_MSG)

    return K

---
### 3단계. 추측을 옮기는 한 줄 / Step 3. The line that moves the guess

덱 「The Reading Is 14. Where Should Our Guess Go?」: 새 추측은 예측에서 출발해 관측 쪽으로 이득만큼 움직입니다. 관측과 예측의 차이 $y - a^-$ 를 덱은 놀람(surprise)이라고 불렀습니다.
*"The Reading Is 14. Where Should Our Guess Go?": the new guess starts at the prediction and moves toward the reading by the gain. The deck called the gap $y - a^-$ between the reading and the prediction the surprise.*

$$a = a^- + K\,(y - a^-)$$

**TODO 2** 는 이 한 줄입니다. 손계산 예: 예측 10, 관측 14, $K = 3/7$ 이면 $10 + (3/7) \times 4 = 11.71$.
***TODO 2** is this line. By hand: with prediction 10, reading 14 and $K = 3/7$, $10 + (3/7) \times 4 = 11.71$.*

**코드를 돌리기 전에 예측하세요.** 이득이 0 이면 새 추측은 어디일까요? 이득이 1 이면?
***Predict before you run.** Where is the new guess when the gain is 0? And when it is 1?*

In [ ]:
# ---------------------------------------------------------------- TODO 2 ----
# 다음 이름을 씁니다 / Names used below
#   a_pred   : 관측을 보기 전의 추측(예측) / the guess before the reading (the prediction)
#   K        : 이득, TODO 1 에서 구한 값 / the gain from TODO 1
#   reading  : 새로 들어온 관측 하나 / the one new reading
#   a_filt   : 관측을 본 뒤의 새 추측   <- 여기가 빈칸
#              the new guess after the reading   <- this is the blank


def move_guess(a_pred, K, reading):
    """덱 「The Reading Is 14. Where Should Our Guess Go?」 의 한 줄 / the one line on that slide."""
    # TODO 2. 예측에서 출발해, 관측과 예측의 차이(놀람)에 K 를 곱한 만큼 옮기세요. 한 줄입니다.
    # TODO 2. Start at the prediction and move by K times the gap between the reading and the prediction (the surprise). One line.
    a_filt = None

    if a_filt is None:
        raise NotImplementedError(BLANK_MSG)

    return a_filt

In [ ]:
# 자가검증 / Self-check   덱 「The Reading Is 14. Where Should Our Guess Go?」 의 표 / the table on that slide
got = move_guess(10, 3 / 7, 14)
assert np.isclose(got, 10 + 12 / 7), (
    f"예측 10, 관측 14, K = 3/7 에서 정답은 11.7143, 결과는 {got:.4f} 입니다. "
    "6.0000 이면 K × 관측만 썼고, 16.0000 이면 예측에 K × 관측을 더했고(차이가 아니라), 12.2857 이면 예측과 관측의 무게가 바뀌었습니다.\n"
    f"With prediction 10, reading 14 and K = 3/7 the answer is 11.7143, but got {got:.4f}. "
    "6.0000 means K × reading alone; 16.0000 means K × reading was added to the prediction (not the gap); 12.2857 means the two weights are swapped.")
table = [round(float(move_guess(10, K, 14)), 2) for K in (0, 0.25, 3 / 7, 0.75, 1)]
assert table == [10.0, 11.0, 11.71, 13.0, 14.0], f"덱의 표는 10, 11, 11.71, 13, 14 인데 결과는 {table} 입니다 / the deck's table has 10, 11, 11.71, 13, 14 but got {table}."
assert np.isclose(move_guess(0, 2 / 3, -1.05), -0.70), "예측 0, K = 2/3, 관측 −1.05 면 −0.70 이어야 합니다 / prediction 0, K = 2/3, reading −1.05 should give −0.70."
print("통과. K = 0, 0.25, 3/7, 0.75, 1 에서 새 추측은", table, ": 덱의 표입니다. K = 0 이면 예측 그대로, K = 1 이면 관측 그대로.")
print("Passed. With K = 0, 0.25, 3/7, 0.75, 1 the new guess is", table, ": the deck's table. K = 0 keeps the prediction, K = 1 takes the reading.")
# 기대값 / Expected   [10.0, 11.0, 11.71, 13.0, 14.0]

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.


def move_guess(a_pred, K, reading):
    """덱 「The Reading Is 14. Where Should Our Guess Go?」 의 한 줄 / the one line on that slide."""
    a_filt = a_pred + K * (reading - a_pred)         # 빈칸이었던 줄. 관측 쪽으로 K 만큼 / the blank: a share K of the way toward the reading

    if a_filt is None:
        raise NotImplementedError(BLANK_MSG)

    return a_filt

---
### 4단계. 한 걸음, 그리고 관측이 오지 않을 때 / Step 4. One step, and what if no reading arrives?

덱 「The Whole Filter Is Two Steps, Repeated」: 한 걸음은 **예측**(추측은 그대로 넘어오고, 불확실성은 $Q$ 만큼 커진다)과 **갱신**(이득을 구하고, 추측을 옮기고, 불확실성을 줄인다)입니다. 아래 `kalman_step` 이 그 한 걸음이고, 2 · 3단계에서 만든 두 함수를 씁니다.
*"The Whole Filter Is Two Steps, Repeated": one step is **predict** (the guess carries over and the uncertainty grows by $Q$) and **update** (find the gain, move the guess, shrink the uncertainty). `kalman_step` below is that one step, and it uses the two functions from steps 2 and 3.*

덱 「What If a Reading Never Arrives?」: 관측이 없으면 놀람도 없으니 갱신을 건너뜁니다. 추측은 예측 그대로입니다. **TODO 3** 은 그때의 불확실성 한 줄입니다.
*"What If a Reading Never Arrives?": with no reading there is no surprise, so we skip the update. The guess stays at the prediction. **TODO 3** is the line for the uncertainty in that case.*

함수가 돌려주는 `K`는 관측이 왔다면 쓸 가중치입니다. 관측이 없는 걸음에서는 계산된 `K`를 갱신에 쓰지 않습니다. 6단계 하단 그래프도 이 잠재적 가중치를 보여 줍니다. 실제 갱신은 관측이 돌아오는 걸음 60에서 다시 시작합니다.
*The returned `K` is the weight we would use if a reading arrived. With a missing reading, the calculated `K` is not used for an update. The lower graph in step 6 shows this potential weight too. Actual updates resume when the reading returns at step 60.*

**코드를 돌리기 전에 예측하세요.** 관측이 두 걸음 연속으로 오지 않으면 불확실성 $P$ 는 커질까요, 그대로일까요, 줄어들까요?
***Predict before you run.** If no reading arrives for two steps in a row, does the uncertainty $P$ grow, stay the same, or shrink?*

In [ ]:
# ---------------------------------------------------------------- TODO 3 ----
# 다음 이름을 씁니다 / Names used below
#   a, P            : 지난 걸음이 끝났을 때의 추측과 그 오차 분산 / the guess and its error variance at the end of the last step
#   reading         : 이번 걸음의 관측. 오지 않았으면 nan(값 없음) / this step's reading; nan (no value) if none arrived
#   s_eta2, s_eps2  : 덱의 Q 와 H / Q and H on the deck
#   a_pred, P_pred  : 관측을 보기 전의 추측과 그 오차 분산 / the guess and its error variance before the reading
#   a_filt, P_filt  : 이번 걸음이 끝났을 때의 추측과 그 오차 분산. 관측이 없을 때의 P_filt   <- 여기가 빈칸
#                     the guess and its error variance at the end of this step. P_filt when no reading arrived   <- this is the blank
#
# 참고 / Reference   np.isnan(x)
#   x 가 nan(값 없음)이면 True, 숫자면 False / True when x is nan (no value), False when it is a number
#   예 / e.g.   np.isnan(np.nan)  ->  True      np.isnan(14)  ->  False


def kalman_step(a, P, reading, s_eta2, s_eps2):
    """한 걸음: 예측, 그리고 관측이 있으면 갱신 / one step: predict, then update if a reading arrived."""
    # 예측 / Predict   (덱 「Before the Reading Arrives, What Do We Expect?」)
    a_pred = a                                        # 추측은 그대로 넘어온다 / the guess carries over
    P_pred = P + s_eta2                               # 참값이 움직였을 수 있으니 불확실성이 Q 만큼 커진다 / the truth may have moved, so the uncertainty grows by Q
    K = kalman_gain(P_pred, s_eps2)                   # TODO 1: 관측이 왔다면 쓸 이득 / TODO 1: the gain if a reading arrives

    # 갱신 / Update
    if np.isnan(reading):                             # 관측이 오지 않았다 / no reading arrived
        a_filt = a_pred                               # 놀람이 없으니 추측은 예측 그대로 / no surprise, so the guess stays at the prediction
        # TODO 3. 갱신을 건너뛰었습니다. 이 걸음이 끝났을 때의 불확실성은 위의 어느 값인가요? 한 줄입니다.
        # TODO 3. The update was skipped. Which value above is the uncertainty at the end of this step? One line.
        P_filt = None

        if P_filt is None:
            raise NotImplementedError(BLANK_MSG)
    else:                                             # 관측이 왔다 / a reading arrived
        a_filt = move_guess(a_pred, K, reading)       # TODO 2: 관측 쪽으로 K 만큼 / toward the reading by K
        P_filt = (1 - K) * P_pred                     # 관측을 썼으니 불확실성이 줄어든다 / a reading was used, so the uncertainty shrinks

    return a_pred, P_pred, K, a_filt, P_filt

In [ ]:
# 자가검증 / Self-check   덱 「What If a Reading Never Arrives?」 의 표 / the table on that slide (Q = 1, H = 4)
a_pred_, P_pred_, K_, a_, P_ = kalman_step(10, 2, 14, 1, 4)                  # 관측이 있는 걸음: 덱의 손계산 / a step with a reading: the deck's worked example
assert np.allclose([a_pred_, P_pred_, K_, a_, P_], [10, 3, 3 / 7, 10 + 12 / 7, 12 / 7]), (
    "관측이 있는 걸음이 덱의 손계산(예측 10 과 3, 이득 3/7, 새 추측 11.71, 새 P 1.71)과 다릅니다. TODO 1 과 TODO 2 의 자가검증을 다시 실행해 보세요.\n"
    "The step with a reading differs from the deck's worked example (prediction 10 and 3, gain 3/7, new guess 11.71, new P 1.71). Run the self-checks of TODO 1 and TODO 2 again.")
rows = []
for reading in (np.nan, np.nan, 13):                                         # 관측 없음, 관측 없음, 관측 13 / no reading, no reading, then the reading 13
    a_pred_, P_pred_, K_, a_, P_ = kalman_step(a_, P_, reading, 1, 4)
    rows.append((reading, P_pred_, K_, a_, P_))
P_after = [round(float(r[4]), 2) for r in rows]
assert P_after[:2] == [2.71, 3.71], (
    f"관측이 없는 두 걸음 뒤의 P 는 2.71, 3.71 이어야 하는데 결과는 {P_after[:2]} 입니다. "
    "[1.71, 1.71] 이면 지난 P 를 그대로 뒀고(예측에서 커진 것을 잃음), [1.62, 1.58] 이면 관측이 없는데도 (1 − K) 를 곱해 줄였습니다.\n"
    f"After two steps with no reading P should be 2.71 and 3.71, but got {P_after[:2]}. "
    "[1.71, 1.71] means the old P was kept (the growth in the predict step was lost); [1.62, 1.58] means it was shrunk by (1 − K) although no reading arrived.")
assert np.isclose(rows[0][3], 10 + 12 / 7) and np.isclose(rows[1][3], 10 + 12 / 7), "관측이 없는 동안 추측은 11.71 그대로여야 합니다 / the guess should stay at 11.71 while no reading arrives."
assert round(float(rows[2][2]), 2) == 0.54 and round(float(rows[2][3]), 2) == 12.41 and round(float(rows[2][4]), 2) == 2.16, "관측 13 이 돌아온 걸음이 덱의 표(0.54, 12.41, 2.16)와 다릅니다 / the step where the reading 13 returns differs from the deck's table (0.54, 12.41, 2.16)."
print("통과. 관측이 없는 두 걸음: 추측 11.71 그대로, P 는 1.71 -> 2.71 -> 3.71. 돌아온 관측 13 의 이득 0.54(평소 약 0.40), 새 추측 12.41, P 2.16: 덱의 표입니다.")
print("Passed. Two steps with no reading: the guess stays at 11.71 and P goes 1.71 -> 2.71 -> 3.71. The returning reading 13 gets the gain 0.54 (about 0.40 usually), new guess 12.41, P 2.16: the deck's table.")

In [ ]:
#@title ▶ Solution / 정답. 막혔을 때만 실행하세요 (위에서 쓴 것을 덮어씁니다) { display-mode: "form" }
# 위 TODO 셀에서 빈칸 한 줄만 채운 것입니다. 실행 로직은 빈칸 한 줄만 다릅니다.
# The TODO cell with the blank filled in. Only that one line of logic differs.
# 실행한 뒤에는 바로 위의 자가검증 셀을 다시 실행하세요. 자가검증은 지금 메모리에 있는 것을 검사합니다.
# After running it, run the self-check cell just above again. The self-check tests whatever is in memory now.


def kalman_step(a, P, reading, s_eta2, s_eps2):
    """한 걸음: 예측, 그리고 관측이 있으면 갱신 / one step: predict, then update if a reading arrived."""
    # 예측 / Predict   (덱 「Before the Reading Arrives, What Do We Expect?」)
    a_pred = a                                        # 추측은 그대로 넘어온다 / the guess carries over
    P_pred = P + s_eta2                               # 참값이 움직였을 수 있으니 불확실성이 Q 만큼 커진다 / the truth may have moved, so the uncertainty grows by Q
    K = kalman_gain(P_pred, s_eps2)                   # TODO 1: 관측이 왔다면 쓸 이득 / TODO 1: the gain if a reading arrives

    # 갱신 / Update
    if np.isnan(reading):                             # 관측이 오지 않았다 / no reading arrived
        a_filt = a_pred                               # 놀람이 없으니 추측은 예측 그대로 / no surprise, so the guess stays at the prediction
        P_filt = P_pred                               # 빈칸이었던 줄. 예측에서 커진 채로 남는다 / the blank: it stays as large as the prediction left it

        if P_filt is None:
            raise NotImplementedError(BLANK_MSG)
    else:                                             # 관측이 왔다 / a reading arrived
        a_filt = move_guess(a_pred, K, reading)       # TODO 2: 관측 쪽으로 K 만큼 / toward the reading by K
        P_filt = (1 - K) * P_pred                     # 관측을 썼으니 불확실성이 줄어든다 / a reading was used, so the uncertainty shrinks

    return a_pred, P_pred, K, a_filt, P_filt

---
### 5단계. 센서 자료에 필터 돌리기 / Step 5. Run the filter on the sensor data

덱 「Watch the Filter Track the Hidden Value」. 아래 `kalman_filter` 는 4단계의 한 걸음을 자료 끝까지 되풀이할 뿐입니다. 시작은 추측 0 과 아주 큰 불확실성 $P_0 = 10^6$ 입니다. 아직 아는 것이 없다는 뜻입니다(덱 「Where Does the Filter Start?」).
*"Watch the Filter Track the Hidden Value". `kalman_filter` below only repeats step 4's one step to the end of the data. It starts from the guess 0 with a huge uncertainty $P_0 = 10^6$, which says we know nothing yet ("Where Does the Filter Start?").*

**코드를 돌리기 전에 예측하세요** (덱 「In the Lab, Filter, Break the Sensor, Then Smooth」 표의 첫 두 줄).
***Predict before you run** (the first two rows of the table on "In the Lab, Filter, Break the Sensor, Then Smooth").*

1. 첫 이득은 0 에 가까울까요, 1 에 가까울까요? / Is the first gain near 0 or near 1?
2. 이득은 어디에 자리 잡을까요? / Where will the gain settle?

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your predictions here. This cell does not need to be run.
my_prediction_1 = ""   # 첫 이득 / the first gain
my_prediction_2 = ""   # 자리 잡는 이득 / the settled gain

In [ ]:
def kalman_filter(y, s_eta2, s_eps2, a0=0.0, P0=1e6):
    """한 걸음을 자료 끝까지 되풀이 / repeat the one step to the end of the data."""
    n = len(y)                                        # 걸음 수 / the number of steps
    a_pred, P_pred, K, a_filt, P_filt = (np.zeros(n) for _ in range(5))   # 걸음마다의 값을 담을 빈 배열 다섯 / five empty arrays, one value per step
    a, P = a0, P0                                     # 시작: 추측 0, 아주 큰 불확실성 / start: the guess 0 and a huge uncertainty
    for t in range(n):                                # t = 0, 1, ..., n − 1
        a_pred[t], P_pred[t], K[t], a_filt[t], P_filt[t] = kalman_step(a, P, y[t], s_eta2, s_eps2)   # 4단계의 한 걸음 / step 4's one step
        a, P = a_filt[t], P_filt[t]                   # 이번 걸음의 결과가 다음 걸음의 출발 / this step's result is where the next step starts
    return a_pred, P_pred, K, a_filt, P_filt

In [ ]:
a_pred, P_pred, K, a_filt, P_filt = kalman_filter(y, s_eta2, s_eps2)        # 끊기지 않은 센서, 120분 전체 / the unbroken sensor, all 120 minutes
print("처음 세 이득 / the first three gains:", np.round(K[:3], 4))
print(f"걸음 40 의 이득 / the gain at step 40: {K[40]:.3f},   식으로 구한 값 / from the formula: {settled_gain(s_eta2, s_eps2):.3f}")
print(f"처음 45분의 오차(RMSE) / error over the first 45 minutes (RMSE):  최근 관측 / latest reading {rmse(y[first], alpha[first]):.3f},"
      f"  지금까지의 평균 / average so far {rmse(average_so_far[first], alpha[first]):.3f},  필터 / filter {rmse(a_filt[first], alpha[first]):.3f}")
fig, axes = plt.subplots(2, 1, figsize=(10, 5.2), gridspec_kw={"height_ratios": [3, 1.2]}, sharex=True)
axes[0].plot(alpha[first], color=DARK, lw=2, label="true value")
axes[0].plot(y[first], "o", color=GRAY, ms=4, label="readings")
axes[0].plot(a_filt[first], color=GOLD, lw=2, label="Kalman filter")
axes[0].set(ylabel="value", title="The filter follows the truth but ignores most of the noise")   # 그림 글은 영어만 / plot text in English only
axes[0].legend(ncol=3, fontsize=8)
axes[1].plot(K[first], color=GREEN)
axes[1].set(xlabel="step t (minutes)", ylabel="gain K", ylim=(0, 1.05))
plt.tight_layout(); plt.show()
# 기대값 / Expected   처음 세 이득 / the first three gains: [1.     0.5215 0.3795]
# 기대값 / Expected   걸음 40 의 이득 / the gain at step 40: 0.258,   식으로 구한 값 / from the formula: 0.258
# 기대값 / Expected   최근 관측 / latest reading 0.928,  지금까지의 평균 / average so far 0.827,  필터 / filter 0.415

**What to notice.** 첫 이득이 1 인 것은 시작 불확실성이 아주 커서 첫 관측을 거의 그대로 받아들인다는 뜻입니다. 그 뒤 이득은 몇 걸음 만에 0.258 에 자리 잡습니다. 관측이 올 때마다 추측을 약 4분의 1 만큼 옮기는 것입니다. 필터의 오차 0.415 는 최근 관측(0.928)의 절반보다 작습니다. 이득이 자리 잡는 값은 관측값이 아니라 $Q$ 와 $H$ 만으로 정해집니다(덱 「The Settled Gain in One Formula」). 다음 주에는 이 고정된 이득이 지수평활의 가중치가 됩니다.
*The first gain is 1 because the starting uncertainty is huge: the first reading is taken almost as it is. After a few steps the gain settles at 0.258, so each reading moves the guess about a quarter of the way. The filter's error, 0.415, is less than half that of the latest reading (0.928). Where the gain settles depends only on $Q$ and $H$, not on the readings' values ("The Settled Gain in One Formula"). Next week this fixed gain becomes the weight of exponential smoothing.*

---
### 6단계. 센서를 끊기 / Step 6. Break the sensor

덱 「The Filter Across a Gap in the Readings」: 관측 45 ~ 59 를 지웁니다. 지운 자리를 다른 값으로 채우지도, 그 걸음을 빼 버리지도 않습니다. 값 없음(`nan`)으로 두면 4단계의 `kalman_step` 이 갱신을 건너뜁니다.
*"The Filter Across a Gap in the Readings": remove readings 45 to 59. We neither fill the hole with other values nor drop those steps. We leave them as "no value" (`nan`), and step 4's `kalman_step` skips the update.*

**코드를 돌리기 전에 예측하세요** (덱 실습 표의 셋째 줄). 걸음 45 ~ 59 에서 95 % 띠는 어떻게 될까요? 걸음 60 에 관측이 돌아오면 그 이득은 평소(0.258)보다 클까요, 작을까요?
***Predict before you run** (the third row of the deck's lab table). What happens to the 95% band in steps 45 to 59? When the reading returns at step 60, is its gain larger or smaller than usual (0.258)?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_3 = ""   # 결측 구간의 띠와 걸음 60 의 이득 / the band across the gap and the gain at step 60

In [ ]:
y_gap = y.copy()                                           # 사본을 만든다. 원래 관측 y 는 그대로 / work on a copy; the original readings y stay as they are
y_gap[45:60] = np.nan                                      # 관측 45 ~ 59 를 지운다(60 은 들어가지 않는다) / remove readings 45 to 59 (60 is not included)
a_pred, P_pred, K, a_filt, P_filt = kalman_filter(y_gap, s_eta2, s_eps2)   # 같은 필터, 끊긴 센서 / the same filter on the broken sensor
band = 1.96 * np.sqrt(P_filt)                              # 95 % 띠의 반폭 = 1.96 × 표준편차 / half-width of the 95% band = 1.96 × standard deviation
inside = np.abs(alpha - a_filt) <= band                    # 참값이 띠 안에 있는 걸음은 True / True at the steps where the true value is inside the band
print(f"P:  걸음 44 / step 44 = {P_filt[44]:.3f},  걸음 59 / step 59 = {P_filt[59]:.3f}   (15걸음 동안 Q 씩, 15 × {s_eta2:.2f} = {15 * s_eta2:.2f} 커짐 / grew by Q for 15 steps)")
print(f"걸음 60, 관측 직전 / step 60, just before the reading:  P_pred = {P_pred[60]:.3f},  이득 / gain K = {K[60]:.3f}")
print(f"95 % 띠의 반폭 / half-width of the 95% band:  걸음 44 / step 44 = {band[44]:.2f},  걸음 59 / step 59 = {band[59]:.2f}")
print(f"결측 동안의 추측 / the guess during the gap:  걸음 45 / step 45 = {a_filt[45]:.3f},  걸음 59 / step 59 = {a_filt[59]:.3f}")
print(f"참값이 띠 안에 든 걸음 / steps with the true value inside the band: {inside.sum()} / {len(alpha)}")
steps = np.arange(len(y))
fig, axes = plt.subplots(2, 1, figsize=(10, 5.6), gridspec_kw={"height_ratios": [3, 1.2]}, sharex=True)
axes[0].plot(steps, alpha, color=DARK, lw=1.6, label="true value")
axes[0].plot(steps, y_gap, "o", color=GRAY, ms=3, label="readings")
axes[0].plot(steps, a_filt, color=GOLD, lw=2, label="filter")
axes[0].plot(steps, a_filt + band, "--", color=GOLD, lw=1, label="95% band for the true value")
axes[0].plot(steps, a_filt - band, "--", color=GOLD, lw=1)
axes[0].axvspan(44.5, 59.5, color=RED, alpha=0.08)
axes[0].text(52, 13.2, "no readings", color=RED, ha="center", fontsize=9)
axes[0].set(ylabel="value", ylim=(6, 13.6), title="Across the gap the filter only predicts, and its band widens")   # 그림 글은 영어만 / plot text in English only
axes[0].legend(ncol=4, fontsize=8, loc="lower right")
# 곡선은 관측이 왔다면 쓸 가중치입니다. 결측 동안에는 이 K 로 갱신하지 않습니다.
# The curve is the weight if a reading were available; K is not used for an update during the gap.
axes[1].plot(steps, K, color=GREEN)
axes[1].set(xlabel="step t", ylabel="potential gain K", ylim=(0, 1.05),
            title="Gain if a reading were available; no update during the gap")
plt.tight_layout(); plt.show()
# 기대값 / Expected   P:  걸음 44 / step 44 = 0.258,  걸음 59 / step 59 = 1.608   (15 × 0.09 = 1.35)
# 기대값 / Expected   걸음 60, 관측 직전 / step 60, just before the reading:  P_pred = 1.698,  이득 / gain K = 0.629
# 기대값 / Expected   95 % 띠의 반폭 / half-width of the 95% band:  걸음 44 / step 44 = 1.00,  걸음 59 / step 59 = 2.49
# 기대값 / Expected   결측 동안의 추측 / the guess during the gap:  걸음 45 / step 45 = 9.592,  걸음 59 / step 59 = 9.592
# 기대값 / Expected   참값이 띠 안에 든 걸음 / steps with the true value inside the band: 118 / 120

**What to notice.** 관측이 없는 동안 필터는 예측만 합니다. 추측은 9.592 에 머물고, $P$ 는 걸음마다 $Q = 0.09$ 씩 커져 0.258 에서 걸음 59 에 1.608 이 됩니다. 걸음 60 의 관측 직전에는 한 번 더 커져 $P^- = 1.698$ 이고, 그래서 돌아온 관측의 이득이 $1.698 / (1.698 + 1) = 0.629$ 입니다(평소 0.258). 오래 못 본 만큼 새 관측을 더 믿는 것입니다. 띠는 **참값**의 95 % 띠입니다. 아직 오지 않은 관측의 띠라면 분산에 $H$ 를 더해야 합니다. 참값이 띠 안에 든 걸음이 120 가운데 118 인 것은 한 번의 모의 결과입니다.
*While no reading arrives the filter only predicts. The guess stays at 9.592 and $P$ grows by $Q = 0.09$ each step, from 0.258 to 1.608 at step 59. Just before the reading at step 60 it grows once more, to $P^- = 1.698$, so the returning reading gets the gain $1.698 / (1.698 + 1) = 0.629$ (0.258 usually): the longer we have gone without a reading, the more we trust the next one. The band is a 95% band for the **true value**. For a reading that has not arrived yet, add $H$ to the variance. The true value being inside the band at 118 of 120 steps is the result of this one simulation.*

---
### 7단계. 스무더와 우도, 읽기만 / Step 7. The smoother and the likelihood, read only

덱 「What Else Does the Same Recursion Give?」: 같은 계산에서 두 가지가 더 나옵니다. **스무더**는 자료를 끝까지 본 뒤 뒤에서 앞으로 한 번 더 지나가며, 나중 관측까지 써서 추측을 고칩니다. **로그우도**는 놀람들의 로그 밀도를 더한 값입니다. $Q$ 와 $H$ 를 모를 때는 이 값이 가장 커지는 $Q$ 와 $H$ 를 고릅니다. 두 함수는 도구 셀에 있고, 오늘은 결과만 읽습니다.
*"What Else Does the Same Recursion Give?": the same computation gives two more things. The **smoother** makes one more pass, backward, after seeing all the data, and corrects each guess using later readings too. The **log-likelihood** is the sum of the surprises' log densities. When $Q$ and $H$ are unknown, we choose the $Q$ and $H$ that make it largest. Both functions are in the tools cell; today we only read their results.*

**코드를 돌리기 전에 예측하세요** (덱 실습 표의 넷째 줄). 관측, 필터, 스무더 가운데 무엇이 참값에 가장 가까울까요?
***Predict before you run** (the fourth row of the deck's lab table). Which is closest to the truth: the readings, the filter or the smoother?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_4 = ""   # 관측, 필터, 스무더 가운데 / among the readings, the filter and the smoother

In [ ]:
a_smooth, P_smooth = kalman_smoother(a_pred, P_pred, a_filt, P_filt)       # 6단계의 필터 결과에서 / from step 6's filter output
seen = ~np.isnan(y_gap)                                                    # 관측이 있는 걸음은 True, 없는 걸음은 False / True at steps with a reading, False without
print(f"로그우도(첫 관측 제외) / log-likelihood (first reading left out): {log_likelihood(y_gap, a_pred, P_pred, s_eps2):.2f}")
print("참값과의 오차(RMSE) / error against the true value (RMSE)")
print(f"  관측이 있는 {seen.sum()}걸음 / the {seen.sum()} steps with a reading:   관측 / readings {rmse(y_gap[seen], alpha[seen]):.3f},  필터 / filter {rmse(a_filt[seen], alpha[seen]):.3f},  스무더 / smoother {rmse(a_smooth[seen], alpha[seen]):.3f}")
print(f"  관측이 없는 {(~seen).sum()}걸음 / the {(~seen).sum()} steps without one:    필터 / filter {rmse(a_filt[~seen], alpha[~seen]):.3f},  스무더 / smoother {rmse(a_smooth[~seen], alpha[~seen]):.3f}")
print(f"  전체 {len(alpha)}걸음 / all {len(alpha)} steps:                필터 / filter {rmse(a_filt, alpha):.3f},  스무더 / smoother {rmse(a_smooth, alpha):.3f}")
zoom = slice(30, 76)                                                       # 결측 구간 앞뒤만 크게 / zoom in around the gap
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(steps[zoom], alpha[zoom], color=DARK, lw=1.6, label="true value")
ax.plot(steps[zoom], y_gap[zoom], "o", color=GRAY, ms=3, label="readings")
ax.plot(steps[zoom], a_filt[zoom], color=GOLD, lw=2, label="filter (readings so far)")
ax.plot(steps[zoom], a_smooth[zoom], color=GREEN, lw=2, label="smoother (later readings too)")
ax.axvspan(44.5, 59.5, color=RED, alpha=0.08)
ax.set(xlabel="step t", ylabel="value", title="The smoother looks back with hindsight")   # 그림 글은 영어만 / plot text in English only
ax.legend(ncol=2, fontsize=8); plt.tight_layout(); plt.show()
# 기대값 / Expected   로그우도(첫 관측 제외) / log-likelihood (first reading left out): -158.12
# 기대값 / Expected   관측이 있는 105걸음 / the 105 steps with a reading:   관측 / readings 0.931,  필터 / filter 0.438,  스무더 / smoother 0.343
# 기대값 / Expected   관측이 없는 15걸음 / the 15 steps without one:    필터 / filter 0.901,  스무더 / smoother 0.684
# 기대값 / Expected   전체 120걸음 / all 120 steps:                필터 / filter 0.519,  스무더 / smoother 0.402

**What to notice.** 같은 105걸음에서 견주면 관측 0.931, 필터 0.438, 스무더 0.343 입니다. 덱 「What Else Does the Same Recursion Give?」 의 표가 이 첫 줄과 마지막 줄입니다. 관측의 오차는 관측이 있는 105걸음에서만 잴 수 있고, 필터와 스무더는 관측이 없는 15걸음에서도 추측을 내므로 120걸음 전체로도 잴 수 있습니다. 스무더가 가장 작은 것은 나중 관측까지 쓰기 때문입니다. 같은 정보로 겨룬 것이 아닙니다. 결측 구간에서 필터는 평평하게 머물고, 스무더는 돌아온 관측 쪽으로 미리 기울어 있습니다. 지금 이 순간의 추측이 필요하면 필터, 자료가 다 모인 뒤에 과거를 다시 볼 때는 스무더입니다. 예측이 얼마나 맞았는지 채점할 때는 둘 다 아니고, 그 관측을 보기 전의 예측 `a_pred` 를 씁니다.
*On the same 105 steps: readings 0.931, filter 0.438, smoother 0.343. The table on "What Else Does the Same Recursion Give?" shows this first line and the last one. The readings' error can only be measured at the 105 steps that have a reading, while the filter and the smoother also give a guess at the 15 steps without one, so they can be scored on all 120. The smoother is smallest because it also uses later readings: the three did not compete on the same information. Across the gap the filter stays flat, while the smoother already leans toward the readings that return. Use the filter when you need the guess right now, and the smoother when you look back after all the data are in. To score how good a forecast was, use neither: use `a_pred`, the prediction made before that reading.*

---
### 8단계. 설정 하나를 바꾸기 / Step 8. Change one choice

덱 실습 표의 마지막 줄과 꼬리줄입니다. 센서가 더 시끄러우면($\sigma_\varepsilon = 3$, 그래서 $H = 9$) 이득은 어디에 자리 잡을까요? 아래 셀은 설정을 하나씩 바꿔 1 ~ 7단계를 다시 돌립니다. 한 줄을 골라 기본과 무엇이 달라졌는지 한 문장으로 적으세요.
*The last row and the footer of the deck's lab table. If the sensor were noisier ($\sigma_\varepsilon = 3$, so $H = 9$), where would the gain settle? The cell below reruns steps 1 to 7 changing one setting at a time. Pick one row and write one sentence on what changed from the default.*

표의 띠 반폭은 **결측 마지막 시점**의 값입니다. 관측 RMSE는 관측이 있는 걸음(seen), 필터와 스무더 RMSE는 전체 120걸음(all)에서 잽니다. 세 방법을 같은 걸음에서 비교하려면 7단계의 공통 관측 시점 행을 보세요.
*The band half-width is measured **at the end of the gap**. Reading RMSE uses observed steps (seen); filter and smoother RMSE use all 120 steps (all). To compare the three methods on the same steps, use the common observed-step row in step 7.*

**코드를 돌리기 전에 예측하세요** (덱 실습 표의 다섯째 줄). $\sigma_\varepsilon = 3$ 이면 자리 잡는 이득은 0.258 보다 클까요, 작을까요?
***Predict before you run** (the fifth row of the deck's lab table). With $\sigma_\varepsilon = 3$, is the settled gain above or below 0.258?*

In [ ]:
# 여기에 예측을 적으세요. 이 셀은 실행하지 않아도 됩니다.
# Write your prediction here. This cell does not need to be run.
my_prediction_5 = ""   # 센서가 더 시끄러울 때 자리 잡는 이득 / the settled gain with a noisier sensor

In [ ]:
print("오차 범위 / RMSE scope: 관측 / readings = 관측이 있는 걸음 / observed steps (seen); 필터·스무더 / filter and smoother = 전체 120걸음 / all 120 steps (all)")
print("설정 / setting                    자리 잡은 이득   결측 뒤 첫 이득    결측 끝 반폭     RMSE: 관측(seen)  필터(all)  스무더(all)")
print("                                  settled gain     first gain after   gap-end half-width     readings(seen)  filter(all)  smoother(all)")
for label, kw in (("기본 / default                 ", {}),
                  ("sigma_eps = 3  (H = 9)         ", {"sigma_eps": 3.0}),
                  ("결측 / gap 45 ~ 74 (30 steps)  ", {"gap": (45, 75)}),
                  ("seed 7                         ", {"seed": 7})):
    settled, after, widest, r_read, r_filt, r_smooth = run_lab(**kw)
    print(f"{label}    {settled:.3f}            {after:.3f}              ±{widest:.2f}                 {r_read:.3f}           {r_filt:.3f}    {r_smooth:.3f}")
print(f"식으로 구한 자리 잡는 이득 / the settled gain by formula:  H = 1 -> {settled_gain(0.09, 1.0):.3f},   H = 9 -> {settled_gain(0.09, 9.0):.3f}")
# 기대값 / Expected   기본 / default                     0.258   0.629   ±2.49   0.931   0.519   0.402
# 기대값 / Expected   sigma_eps = 3  (H = 9)             0.095   0.203   ±2.91   2.792   0.949   0.658
# 기대값 / Expected   결측 / gap 45 ~ 74 (30 steps)      0.258   0.753   ±3.37   0.950   0.517   0.574
# 기대값 / Expected   seed 7                             0.258   0.629   ±2.49   0.961   0.514   0.384
# 기대값 / Expected   H = 1 -> 0.258,   H = 9 -> 0.095
#   오차 가운데 관측은 관측이 있는 걸음에서, 필터와 스무더는 120걸음 전체에서 잰 값입니다(7단계의 첫 줄과 마지막 줄).
#   Of the errors, the readings' is over the steps with a reading; the filter's and the smoother's are over all 120 steps (the first and last lines of step 7).

**What to notice.**
- **더 시끄러운 센서.** 이득이 0.258 에서 0.095 로 내려갑니다. 관측 하나가 추측을 10분의 1 도 못 움직입니다. 관측을 덜 믿고 예측을 더 믿는 것입니다.
  *A noisier sensor. The gain drops from 0.258 to 0.095: one reading moves the guess by less than a tenth. We trust the readings less and the prediction more.*
- **다른 시드.** 관측값이 전부 달라졌는데도 이득(0.258, 0.629)과 띠(±2.49)는 기본과 똑같습니다. 이득과 띠는 관측값이 아니라 $Q$, $H$, 그리고 어느 걸음에 관측이 있었는지만으로 정해집니다.
  *Another seed. Every reading changed, yet the gains (0.258, 0.629) and the band (±2.49) are exactly those of the default. The gain and the band depend on $Q$, $H$ and on which steps had a reading, not on the readings' values.*
- **더 긴 결측.** 띠가 ±3.37 까지 넓어지고, 돌아온 관측의 이득이 0.753 입니다. 이 한 번의 모의에서는 스무더(0.574)가 필터(0.517)보다 나쁘게 나왔습니다. 스무더가 평균적으로 더 낫다는 것과 한 번의 결과는 다른 이야기입니다. 아래 「더 해 보기」 의 둘째 셀에서 여러 번 되풀이해 확인합니다.
  *A longer gap. The band widens to ±3.37 and the returning reading gets the gain 0.753. In this one simulation the smoother (0.574) came out worse than the filter (0.517). That the smoother is better on average is a different statement from the result of one run; the second cell of the optional part below repeats the run many times to check.*

---
### 더 해 보기 / Optional

시간이 남거나 집에서 볼 때. 첫 셀은 덱 「How Quickly Does the Gain Settle?」 의 그림을 다시 만듭니다: 손계산 예($Q = 1$, $H = 4$)에서 시작 불확실성이 10, 2, 0.1 이어도 이득은 같은 값에 자리 잡습니다. 둘째 셀은 8단계의 긴 결측을 시드 1,000개로 되풀이해, 스무더가 필터보다 나은 경우가 얼마나 되는지 셉니다.
*For spare time, or at home. The first cell redraws the figure of "How Quickly Does the Gain Settle?": in the worked example ($Q = 1$, $H = 4$) the gain settles at the same value whether the starting uncertainty is 10, 2 or 0.1. The second cell repeats step 8's longer gap with 1,000 seeds and counts how often the smoother beats the filter.*

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.2))
for P0, col in ((10, RED), (2, GOLD), (0.1, GREEN)):
    P, gains = P0, []
    for _ in range(10):                                    # 열 걸음 / ten steps
        _, _, K_one, _, P = kalman_step(0.0, P, 0.0, 1, 4) # 이득과 P 는 관측값과 무관하므로 관측에 0 을 넣었다 / the gain and P do not depend on the reading, so 0 stands in for it
        gains.append(K_one)
    ax.plot(range(1, 11), gains, "o-", color=col, ms=4, label=f"starting P = {P0}")
    print(f"시작 / starting P = {P0:>4}:  처음 세 이득 / first three gains {np.round(gains[:3], 4)},  열째 / tenth {gains[-1]:.4f}")
ax.set(xlabel="step", ylabel="gain K", title="The gain forgets where it started")   # 그림 글은 영어만 / plot text in English only
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print(f"식으로 구한 값 / from the formula: {settled_gain(1, 4):.4f}")
# 기대값 / Expected   시작 / starting P =   10:  처음 세 이득 / first three gains [0.7333 0.4958 0.4272],  열째 / tenth 0.3904
# 기대값 / Expected   시작 / starting P =    2:  처음 세 이득 / first three gains [0.4286 0.4043 0.3955],  열째 / tenth 0.3904
# 기대값 / Expected   시작 / starting P =  0.1:  처음 세 이득 / first three gains [0.2157 0.3177 0.3621],  열째 / tenth 0.3904
# 기대값 / Expected   식으로 구한 값 / from the formula: 0.3904

In [ ]:
better = []
for seed in range(1000):                                   # 시드 0 ~ 999 / seeds 0 to 999
    _, _, _, _, r_filt, r_smooth = run_lab(gap=(45, 75), seed=seed)
    better.append(r_smooth < r_filt)                       # 스무더의 오차가 더 작으면 True / True when the smoother's error is smaller
print(f"긴 결측에서 스무더가 필터보다 나은 모의의 비율 / share of runs where the smoother beats the filter, longer gap: {np.mean(better):.3f}")
# 기대값 / Expected   0.888
#   1,000번 가운데 약 89 % 에서 스무더가 낫습니다. 8단계의 시드 11 은 나머지 11 % 에 든 경우였습니다. 한 번의 모의로 순위를 단정하지 않습니다.
#   The smoother wins in about 89% of the 1,000 runs. Step 8's seed 11 was one of the other 11%. One simulation does not settle a ranking.

---
### 이번 주에 가져갈 것 / What to take away

덱 「Today in Three Sentences」 의 세 문장을 이 노트북의 단계로 다시 적었습니다.
*The three sentences of "Today in Three Sentences", tied to the steps of this notebook.*

1. 상태공간모형에는 움직이는 숨은 값과, 잡음을 거쳐 그 값을 보는 관측이 있습니다. 두 분산이 $Q$ 와 $H$ 입니다(1단계).
   *A state-space model has a hidden value that moves and readings that see it through noise, with variances $Q$ and $H$ (step 1).*
2. 칼만 필터는 예측한 뒤, 이득 $K = P^- / (P^- + H)$ 만큼 관측 쪽으로 움직입니다(2 ~ 5단계, TODO 1 · 2).
   *The Kalman filter predicts, then moves toward each reading by the gain $K = P^- / (P^- + H)$ (steps 2 to 5, TODO 1 and 2).*
3. 관측이 오지 않으면 갱신을 건너뜁니다. 지역수준모형에서는 추측이 그대로이고, $P$ 가 $Q$ 씩 커지고, 다음 관측을 더 크게 셉니다(4 · 6단계, TODO 3).
   *A missing reading means skipping the update: in the local level model the guess stays, $P$ grows by $Q$, and the next reading counts more (steps 4 and 6, TODO 3).*

**다음으로.** 과제 3 의 Part C 는 오늘의 두 걸음에서 출발합니다. 다음 주에는 5단계에서 자리 잡은 이득이 지수평활의 가중치로 돌아옵니다. 12주차 실습의 첫 TODO 는 오늘 TODO 2 와 같은 한 줄을 가중평균 꼴(새 값에 $\alpha$, 지난 수준에 $1 - \alpha$)로 쓴 것입니다.
*Next. Part C of Assignment 3 builds on today's two steps. Next week the gain that settled in step 5 comes back as the weight of exponential smoothing: the first TODO of the Week 12 lab is today's TODO 2 written as a weighted average ($\alpha$ on the new value, $1 - \alpha$ on the last level).*

In [ ]:
# 연습 공간 / Scratch space

In [ ]:
# 실행 환경 기록. 오류를 게시판에 남길 때 이 출력도 함께 붙여 주세요.
# A record of the environment. Paste this output too when you post an error on the board.
import platform, datetime
print("date          :", datetime.datetime.now().isoformat(timespec="seconds"))
print("python        :", sys.version.split()[0], "on", platform.platform())
print("numpy         :", np.__version__, "· matplotlib", matplotlib.__version__)
print("settings      : seed 11 · 120 steps · sigma_eta", sigma_eta, "· sigma_eps", sigma_eps, "· gap 45 to 59")